In [1]:
# %%
# %%
!pip install --upgrade onnx
!pip install --upgrade onnx onnxruntime
!pip install --upgrade onnx onnx-simplifier
!pip install --upgrade onnx onnx-simplifier onnxruntime
# %%


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 109.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 122.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 137.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.24.4
    Uninstalling protobuf-4.24.4:
      Successfully uninstalled protobuf-4.24.4
  Attempting uninstall: onnx
    Found existing installation: onnx 1.15.0rc2
    Uninstalling onnx-1.15.0rc2:
      Successfully uninstalled onnx-1.15.0rc2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 24.2.0 requires protobuf<5,>=4.21, but you have protobuf 6.33.5 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run:

In [2]:

import os
import re
import random
import shutil
import numpy as np
import onnx
from onnx import numpy_helper
import onnxruntime
from onnxsim import simplify
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat, quant_pre_process
from onnxruntime.quantization.calibrate import CalibrationMethod
from PIL import Image
from torchvision import transforms
from collections import defaultdict
# %%
# %%
BASE_DIR = os.getcwd()
INPUT_MODEL_PATH = os.path.join(BASE_DIR, "weights", "yolop-640-640-normalized.onnx")
OUTPUT_DIR = os.path.join(BASE_DIR, "weights")
OUTPUT_MODEL_PATH = os.path.join(OUTPUT_DIR, "yolop-640-640-normalized-quantized-int8-Entropy-Optimized.onnx")
CALIBRATION_IMG_DIR = os.path.join(BASE_DIR, "dataset", "images", "val")
TARGET_SIZE = (640, 640)
ENABLE_REDUCE_RANGE = True
CALIBRATION_METHOD = CalibrationMethod.Entropy

CALIBRATION_IMAGES = ['b2932832-df9384eb.jpg', 'b2afce7f-0cb9967a.jpg', 'b3770858-ea33d0dc.jpg', 'b4946e87-f31e29b1.jpg', 'b49a272b-c52895c3.jpg', 'b5183558-782ad31c.jpg', 'b5770084-9b15155d.jpg', 'b5a50c2d-ccafe43a.jpg', 'b64e5796-3aa7adc0.jpg', 'b780088d-b85f1b00.jpg', 'b792daf1-e95167d2.jpg', 'b7d13f97-3df013fa.jpg', 'b7f43f41-b1354d92.jpg', 'b90d0148-bbecdb11.jpg', 'b9d12c48-d855b440.jpg', 'ba65317f-d4aba330.jpg', 'bb3bb8c0-2ef3abc6.jpg', 'bb6e9fc4-35369ff6.jpg', 'bcd8fde8-51118aa7.jpg', 'bcf7f6b9-88bfed8f.jpg', 'bd5f8f37-b3008fa1.jpg', 'be776742-fe6599b2.jpg', 'bea82122-66fc59a4.jpg', 'bf1a30f8-59a115d8.jpg', 'c019c1f1-883ea964.jpg', 'c0c9ec9a-dad1653f.jpg', 'c1ba5ee6-1a00a269.jpg', 'c2341eec-16d212b8.jpg', 'c28cecbb-11796cf6.jpg', 'c3698cec-77311999.jpg', 'c3a18207-504667c7.jpg', 'c4f4718b-5356b324.jpg', 'c54ec43c-1aa7875b.jpg', 'c57cda5b-7ca48687.jpg', 'c5b2506d-9121123c.jpg', 'c6d51f76-d2459db8.jpg', 'c7346130-9d225646.jpg', 'c75e23fe-fbadb3fa.jpg', 'c891763b-fea20b04.jpg', 'c962fce0-dcea4e6a.jpg']
# %%
# %%
class YOLOPDataReader(CalibrationDataReader):
    
    def __init__(self, image_folder, model_path, image_list):
        self.image_folder = image_folder
        self.enum_data_dicts = None
        self.datasize = 0
        session = onnxruntime.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        self.input_name = session.get_inputs()[0].name
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        self.image_files = []
        for img_name in image_list:
            img_path = os.path.join(image_folder, img_name)
            if os.path.exists(img_path):
                self.image_files.append(img_name)
            else:
                print(f"ATTENTION : Image introuvable : {img_name}")
        
        self.datasize = len(self.image_files)
        print(f"DataReader : {self.datasize}/{len(image_list)} images de calibration chargees")
    
    def letterbox_image(self, image, size):
        iw, ih = image.size
        w, h = size
        scale = min(w/iw, h/ih)
        nw = int(iw * scale)
        nh = int(ih * scale)
        
        image = image.resize((nw, nh), Image.BICUBIC)
        new_image = Image.new('RGB', size, (114, 114, 114))
        new_image.paste(image, ((w - nw) // 2, (h - nh) // 2))
        return new_image
    
    def get_next(self):
        if self.enum_data_dicts is None:
            self.enum_data_dicts = iter(self.load_batch())
        try:
            return next(self.enum_data_dicts)
        except StopIteration:
            return None
    
    def load_batch(self):
        for filename in self.image_files:
            image_path = os.path.join(self.image_folder, filename)
            image = Image.open(image_path).convert("RGB")
            image = self.letterbox_image(image, TARGET_SIZE)
            
            input_tensor = self.normalize(image)
            input_numpy = input_tensor.unsqueeze(0).numpy()
            
            yield {self.input_name: input_numpy}
# %%
# %%
def force_bn_fusion(model_path, output_path):
    print(f"  Tentative de fusion forcee avec shapes statiques...")
    model = onnx.load(model_path)
    
    input_name = model.graph.input[0].name
    input_shapes = {input_name: [1, 3, 640, 640]}
    
    bn_count_before = sum(1 for node in model.graph.node if node.op_type == 'BatchNormalization')
    
    try:
        model_sim, check = simplify(model, overwrite_input_shapes=input_shapes)
        
        if check:
            bn_count_after = sum(1 for node in model_sim.graph.node if node.op_type == 'BatchNormalization')
            print(f"  Fusion BatchNorm : {bn_count_before} -> {bn_count_after} nodes restantes")
            
            if bn_count_after > 0:
                print("  ATTENTION : Des BatchNorms subsistent. Le modele est peut-etre complexe.")
            else:
                print("  SUCCES : Tous les BatchNorms ont ete fusionnes.")
                
            onnx.save(model_sim, output_path)
            return output_path
        else:
            print(f"  Simplification echouee (Check invalide), utilisation du modele original")
            return model_path
            
    except Exception as e:
        print(f"  Erreur lors de la fusion : {e}")
        return model_path
# %%
# %%
def detect_per_channel_incompatible_weights(model_path, log_output=True):
    """
    Detecte les poids incompatibles avec la quantisation per-channel
    en analysant les warnings typiques (rank 1, axis 1 invalide)
    """
    model = onnx.load(model_path)
    incompatible_weights = []
    
    for initializer in model.graph.initializer:
        weight_name = initializer.name
        dims = initializer.dims
        rank = len(dims)
        
        # Les poids de BatchNorm ont typiquement rank=1 et causent des problemes avec per-channel (axis=1)
        if rank == 1 and ('.bn.weight' in weight_name or '.bn.bias' in weight_name):
            incompatible_weights.append(weight_name)
            if log_output:
                print(f"  Poids incompatible per-channel detecte : {weight_name} (rank={rank})")
    
    if log_output:
        print(f"Total : {len(incompatible_weights)} poids incompatibles per-channel identifies")
    
    return incompatible_weights
# %%
# %%
def merge_quantized_models(perchannel_model_path, pertensor_model_path, output_path):
    """
    Fusionne deux modeles quantises : 
    - Un avec quantisation per-channel (toutes couches sauf BatchNorm)
    - Un avec quantisation per-tensor (uniquement BatchNorm)
    """
    print(f"  Chargement des deux modeles quantises...")
    model_perchannel = onnx.load(perchannel_model_path)
    model_pertensor = onnx.load(pertensor_model_path)
    
    # Recuperer les initializers du modele per-tensor (BatchNorm quantises)
    pertensor_inits = {init.name: init for init in model_pertensor.graph.initializer}
    
    # Remplacer dans le modele per-channel les initializers BatchNorm par ceux du per-tensor
    merged_count = 0
    for i, init in enumerate(model_perchannel.graph.initializer):
        if '.bn.' in init.name and init.name in pertensor_inits:
            model_perchannel.graph.initializer[i].CopyFrom(pertensor_inits[init.name])
            merged_count += 1
    
    print(f"  {merged_count} poids BatchNorm fusionnes")
    
    # Sauvegarder le modele fusionne
    onnx.save(model_perchannel, output_path)
    print(f"  Modeles fusionnes avec succes")
# %%
# %%
def analyze_model_dtypes(model_path):
    """
    Analyse complete des types de donnees dans le modele ONNX
    Retourne un rapport detaille par type et par operation
    """
    model = onnx.load(model_path)
    
    # Mapping ONNX dtype -> nom lisible
    dtype_map = {
        1: 'fp32',
        2: 'uint8',
        3: 'int8',
        4: 'uint16',
        5: 'int16',
        6: 'int32',
        7: 'int64',
        10: 'fp16',
        11: 'fp64',
        12: 'uint32',
        13: 'uint64'
    }
    
    stats = defaultdict(lambda: {'count': 0, 'ops': defaultdict(int)})
    node_types_by_dtype = defaultdict(lambda: defaultdict(int))
    
    # Analyse des initializers (poids)
    for initializer in model.graph.initializer:
        dtype = initializer.data_type
        dtype_name = dtype_map.get(dtype, f'unknown_{dtype}')
        stats[dtype_name]['count'] += 1
        stats[dtype_name]['ops']['Initializer'] += 1
    
    # Analyse des nodes (operations) - on veut savoir QUEL TYPE DE NODE
    quantize_nodes = defaultdict(int)
    for node in model.graph.node:
        # Pour les nodes QuantizeLinear et DequantizeLinear
        if node.op_type in ['QuantizeLinear', 'DequantizeLinear']:
            stats['int8_ops']['count'] += 1
            stats['int8_ops']['ops'][node.op_type] += 1
            quantize_nodes[node.op_type] += 1
    
    # Analyse des value_info (types intermediaires) - associer au type de node qui les produit
    # On cree un mapping output_name -> node_type
    output_to_node = {}
    for node in model.graph.node:
        for output in node.output:
            output_to_node[output] = node.op_type
    
    for value_info in model.graph.value_info:
        if value_info.type.HasField('tensor_type'):
            dtype = value_info.type.tensor_type.elem_type
            dtype_name = dtype_map.get(dtype, f'unknown_{dtype}')
            stats[dtype_name]['count'] += 1
            
            # Trouver le type de node qui produit cette value
            node_type = output_to_node.get(value_info.name, 'Unknown')
            stats[dtype_name]['ops'][node_type] += 1
            node_types_by_dtype[dtype_name][node_type] += 1
    
    # Ajouter les infos sur les nodes de quantization
    stats['quantization_nodes'] = quantize_nodes
    stats['node_types_by_dtype'] = node_types_by_dtype
    
    return stats
# %%
# %%
def print_dtype_analysis(stats):
    """
    Affiche le rapport d'analyse des types de donnees
    """
    print("\n" + "=" * 70)
    print("  ANALYSE DES TYPES DE DONNEES DU MODELE")
    print("=" * 70)
    
    if not stats:
        print("Aucune donnee trouvee dans le modele")
        return
    
    # Extraire les infos supplementaires
    quantization_nodes = stats.pop('quantization_nodes', {})
    node_types_by_dtype = stats.pop('node_types_by_dtype', {})
    
    total_elements = sum(stat['count'] for stat in stats.values())
    
    print(f"\nTotal d'elements analyses : {total_elements}")
    print("\nRepartition par type de donnees :")
    print("-" * 70)
    
    # Tri par nombre d'occurrences
    sorted_stats = sorted(stats.items(), key=lambda x: x[1]['count'], reverse=True)
    
    for dtype_name, data in sorted_stats:
        count = data['count']
        percentage = (count / total_elements * 100) if total_elements > 0 else 0
        print(f"\n{dtype_name.upper()} : {count} elements ({percentage:.1f}%)")
        
        # Detail par type d'operation
        if data['ops']:
            print(f"  Repartition par type :")
            for op_type, op_count in sorted(data['ops'].items(), key=lambda x: x[1], reverse=True):
                print(f"    - {op_type}: {op_count}")
        
        # Ajouter les types de nodes pour les ValueInfo
        if dtype_name in node_types_by_dtype and dtype_name != 'int8_ops':
            print(f"  Types de nodes produisant ce dtype :")
            for node_type, node_count in sorted(node_types_by_dtype[dtype_name].items(), key=lambda x: x[1], reverse=True):
                print(f"    - {node_type}: {node_count}")
    
    print("\n" + "=" * 70)
    
    # Verification de la quantisation complete
    int8_count = stats.get('int8', {}).get('count', 0)
    uint8_count = stats.get('uint8', {}).get('count', 0)
    int8_ops_count = stats.get('int8_ops', {}).get('count', 0)
    fp32_count = stats.get('fp32', {}).get('count', 0)
    fp16_count = stats.get('fp16', {}).get('count', 0)
    int32_count = stats.get('int32', {}).get('count', 0)
    int64_count = stats.get('int64', {}).get('count', 0)
    
    print("\nRESUME DE LA QUANTISATION :")
    print(f"  Elements quantises (INT8/UINT8) : {int8_count + uint8_count}")
    print(f"  Operations de quantisation : {int8_ops_count}")
    if quantization_nodes:
        for op_type, count in quantization_nodes.items():
            print(f"    - {op_type}: {count}")
    
    print(f"\n  Elements FP32 restants : {fp32_count}")
    if fp32_count > 0 and 'fp32' in node_types_by_dtype:
        print(f"    Types de nodes FP32 :")
        for node_type, count in sorted(node_types_by_dtype['fp32'].items(), key=lambda x: x[1], reverse=True)[:10]:
            print(f"      - {node_type}: {count}")
    
    print(f"  Elements FP16 restants : {fp16_count}")
    print(f"  Elements INT32 : {int32_count}")
    print(f"  Elements INT64 : {int64_count}")
    
    if fp32_count == 0 and fp16_count == 0:
        print("\n  ✓ QUANTISATION COMPLETE A 100% !")
    elif fp32_count > 0:
        # Analyser si ce sont juste des ops normales (Reshape, Transpose, etc.)
        if 'fp32' in node_types_by_dtype:
            fp32_node_types = set(node_types_by_dtype['fp32'].keys())
            benign_ops = {'Reshape', 'Transpose', 'Concat', 'Slice', 'Unsqueeze', 'Squeeze', 'Gather', 'Shape', 'ConstantOfShape', 'Unknown'}
            critical_ops = fp32_node_types - benign_ops
            
            if critical_ops:
                print(f"\n  ⚠ ATTENTION : {fp32_count} elements FP32 non quantises")
                print(f"    Operations critiques en FP32 : {critical_ops}")
            else:
                print(f"\n  ℹ INFO : {fp32_count} elements FP32 (operations normales non quantifiables)")
                print(f"    Types : {fp32_node_types}")
    
    print("=" * 70)
# %%
# %%
def quantize_model():
    print("=" * 70)
    print("  QUANTIFICATION YOLOP COMPLETE (100%) - STRATEGIE 2 PASSES")
    print("=" * 70)
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if not os.path.exists(INPUT_MODEL_PATH):
        print(f"ERREUR : Modele introuvable : {INPUT_MODEL_PATH}")
        return
    
    print("\n[1/6] Simplification et fusion forcee Conv+BN...")
    SIMPLIFIED_PATH = os.path.join(OUTPUT_DIR, "temp_simplified.onnx")
    SIMPLIFIED_PATH = force_bn_fusion(INPUT_MODEL_PATH, SIMPLIFIED_PATH)
    
    print(f"\n  Validation et pre-traitement ONNX Runtime...")
    PREPROCESSED_PATH = os.path.join(OUTPUT_DIR, "temp_preprocessed.onnx")
    quant_pre_process(SIMPLIFIED_PATH, PREPROCESSED_PATH)
    
    if os.path.exists(SIMPLIFIED_PATH) and SIMPLIFIED_PATH != INPUT_MODEL_PATH:
        os.remove(SIMPLIFIED_PATH)
        
    SIMPLIFIED_PATH = PREPROCESSED_PATH
    print(f"  Modele pre-traite : {SIMPLIFIED_PATH}")
    
    print("\n[2/6] Detection des poids incompatibles per-channel...")
    incompatible_weights = detect_per_channel_incompatible_weights(SIMPLIFIED_PATH)
    
    print(f"\n[3/6] Preparation des donnees de calibration...")
    print(f"  Source : {CALIBRATION_IMG_DIR}")
    print(f"  Nombre d'images : {len(CALIBRATION_IMAGES)}")
    
    if not os.path.exists(CALIBRATION_IMG_DIR):
        print(f"ERREUR : Dossier de calibration introuvable")
        return
    
    print("\n[4/6] PASSE 1 : Quantisation per-channel (couches compatibles)...")
    print(f"  Format       : QDQ")
    print(f"  Activations  : QUInt8")
    print(f"  Weights      : QInt8")
    print(f"  Per-channel  : Active")
    print(f"  Reduce range : {'Active' if ENABLE_REDUCE_RANGE else 'Desactive'}")
    print(f"  Calibration  : {CALIBRATION_METHOD}")
    print(f"  Poids exclus : {len(incompatible_weights)}")
    
    TEMP_PERCHANNEL_PATH = os.path.join(OUTPUT_DIR, "temp_perchannel.onnx")
    
    try:
        dr1 = YOLOPDataReader(CALIBRATION_IMG_DIR, SIMPLIFIED_PATH, CALIBRATION_IMAGES)
        
        extra_options = {}
        extra_options['WeightSymmetric'] = True
        extra_options['ActivationSymmetric'] = False
        
        if incompatible_weights:
            extra_options['nodes_to_exclude'] = incompatible_weights
        
        quantize_static(
            model_input=SIMPLIFIED_PATH,
            model_output=TEMP_PERCHANNEL_PATH,
            calibration_data_reader=dr1,
            quant_format=QuantFormat.QDQ,
            activation_type=QuantType.QUInt8,
            weight_type=QuantType.QInt8,
            per_channel=True,
            reduce_range=ENABLE_REDUCE_RANGE,
            calibrate_method=CALIBRATION_METHOD,
            extra_options=extra_options
        )
        print(f"  ✓ Passe 1 terminee : {TEMP_PERCHANNEL_PATH}")
        
    except Exception as e:
        print(f"\nERREUR lors de la passe 1 :")
        print(f"    {str(e)}")
        import traceback
        traceback.print_exc()
        return
    
    print("\n[5/6] PASSE 2 : Quantisation per-tensor (couches incompatibles)...")
    print(f"  Per-channel  : Desactive (per-tensor)")
    print(f"  Cible        : {len(incompatible_weights)} poids BatchNorm")
    
    if len(incompatible_weights) > 0:
        try:
            dr2 = YOLOPDataReader(CALIBRATION_IMG_DIR, SIMPLIFIED_PATH, CALIBRATION_IMAGES)
            
            # Pour la passe 2 : on quantise UNIQUEMENT les poids incompatibles
            # On doit donc EXCLURE tous les autres poids
            model_temp = onnx.load(SIMPLIFIED_PATH)
            all_weights = [init.name for init in model_temp.graph.initializer]
            
            # Poids a EXCLURE = tous sauf les incompatibles
            weights_to_exclude_pass2 = [w for w in all_weights if w not in incompatible_weights]
            
            print(f"  Poids a quantiser (per-tensor) : {len(incompatible_weights)}")
            print(f"  Poids a exclure : {len(weights_to_exclude_pass2)}")
            
            extra_options = {}
            extra_options['WeightSymmetric'] = True
            extra_options['ActivationSymmetric'] = False
            extra_options['nodes_to_exclude'] = weights_to_exclude_pass2
            
            TEMP_PERTENSOR_PATH = os.path.join(OUTPUT_DIR, "temp_pertensor.onnx")
            quantize_static(
                model_input=SIMPLIFIED_PATH,
                model_output=TEMP_PERTENSOR_PATH,
                calibration_data_reader=dr2,
                quant_format=QuantFormat.QDQ,
                activation_type=QuantType.QUInt8,
                weight_type=QuantType.QInt8,
                per_channel=False,  # Per-tensor pour cette passe
                reduce_range=ENABLE_REDUCE_RANGE,
                calibrate_method=CALIBRATION_METHOD,
                extra_options=extra_options
            )
            
            print(f"  ✓ Passe 2 terminee : {TEMP_PERTENSOR_PATH}")
            
            # Fusion des deux modeles
            print(f"  Fusion des modeles per-channel et per-tensor...")
            merge_quantized_models(TEMP_PERCHANNEL_PATH, TEMP_PERTENSOR_PATH, OUTPUT_MODEL_PATH)
            
            if os.path.exists(TEMP_PERTENSOR_PATH):
                os.remove(TEMP_PERTENSOR_PATH)
            
            print(f"  ✓ Modeles fusionnes : {OUTPUT_MODEL_PATH}")
            
        except Exception as e:
            print(f"\nERREUR lors de la passe 2 :")
            print(f"    {str(e)}")
            import traceback
            traceback.print_exc()
            return
    else:
        print(f"  Aucun poids incompatible detecte, copie du modele passe 1...")
        shutil.copy(TEMP_PERCHANNEL_PATH, OUTPUT_MODEL_PATH)
    
    print("\n[6/6] Analyse complete du modele quantise...")
    dtype_stats = analyze_model_dtypes(OUTPUT_MODEL_PATH)
    print_dtype_analysis(dtype_stats)
    
    print("\n" + "=" * 70)
    print("  RESULTATS FINAUX DE LA QUANTIFICATION")
    print("=" * 70)
    
    if os.path.exists(OUTPUT_MODEL_PATH):
        size_original = os.path.getsize(INPUT_MODEL_PATH) / (1024 * 1024)
        size_quantized = os.path.getsize(OUTPUT_MODEL_PATH) / (1024 * 1024)
        reduction = size_original / size_quantized
        
        print(f"\nTaille des modeles :")
        print(f"    Original (FP32)  : {size_original:>8.2f} MB")
        print(f"    Quantized (INT8) : {size_quantized:>8.2f} MB")
        print(f"    Reduction        : {reduction:>8.2f}x")
        
        print(f"\nModele INT8 final :")
        print(f"    {OUTPUT_MODEL_PATH}")
        
        print(f"\nCaracteristiques du modele :")
        print(f"    Format : ONNX QDQ (Quantize-DeQuantize)")
        print(f"    Precision : INT8 (100% quantise)")
        print(f"    Strategie : 2 passes (per-channel + per-tensor)")
        print(f"    Couches per-channel : Toutes sauf {len(incompatible_weights)} BatchNorm")
        print(f"    Couches per-tensor : {len(incompatible_weights)} poids BatchNorm")
        
        print(f"\nNettoyage des fichiers temporaires...")
        for temp_file in [SIMPLIFIED_PATH, TEMP_PERCHANNEL_PATH]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
                print(f"    {os.path.basename(temp_file)} supprime")
        
        print("\n" + "=" * 70)
    else:
        print("ERREUR : Le fichier de sortie n'a pas ete cree")
# %%
# %%
if __name__ == "__main__":
    quantize_model()

  QUANTIFICATION YOLOP COMPLETE (100%) - STRATEGIE 2 PASSES

[1/6] Simplification et fusion forcee Conv+BN...
  Tentative de fusion forcee avec shapes statiques...
  Fusion BatchNorm : 12 -> 12 nodes restantes
  ATTENTION : Des BatchNorms subsistent. Le modele est peut-etre complexe.

  Validation et pre-traitement ONNX Runtime...
  Modele pre-traite : /workspace/weights/temp_preprocessed.onnx

[2/6] Detection des poids incompatibles per-channel...
  Poids incompatible per-channel detecte : model.13.bn.bias (rank=1)
  Poids incompatible per-channel detecte : model.13.bn.weight (rank=1)
  Poids incompatible per-channel detecte : model.17.bn.bias (rank=1)
  Poids incompatible per-channel detecte : model.17.bn.weight (rank=1)
  Poids incompatible per-channel detecte : model.2.bn.bias (rank=1)
  Poids incompatible per-channel detecte : model.2.bn.weight (rank=1)
  Poids incompatible per-channel detecte : model.20.bn.bias (rank=1)
  Poids incompatible per-channel detecte : model.20.bn.weigh

  ✓ Passe 1 terminee : /workspace/weights/temp_perchannel.onnx

[5/6] PASSE 2 : Quantisation per-tensor (couches incompatibles)...
  Per-channel  : Desactive (per-tensor)
  Cible        : 24 poids BatchNorm
DataReader : 40/40 images de calibration chargees
  Poids a quantiser (per-tensor) : 24
  Poids a exclure : 252
Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 536
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128
  ✓ Passe 2 terminee : /workspace/weights/temp_pertensor.onnx
  Fusion des modeles per-channel et per-tensor...
  Chargement des deux modeles quantises...
  96 poids BatchNorm fusionnes
  Modeles fusionnes avec succes
  ✓ Modeles fusionnes : /workspace/weights/yolop-640-640-normalized-quantized-int8-Entropy-Optimized.onnx

[6/6] Analyse complete du modele quantise...

  ANALYSE DES TYPES DE DONNEES DU MODELE

Total d'elements analyses : 3193

Repartition par ty